# Video Depth Anything RMSE Evaluation

Notebook version of the Video Depth Anything calibration and evaluation pipeline. It loads the prediction stack and ground-truth stack, skips the first 10 frames because those GT frames are on a different scale in the shared Drive data, calibrates on 5 frames, and evaluates RMSE / AbsRel / delta1 on the remaining frames.

In [ ]:
import sys
import os
from pathlib import Path

import cv2
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit

print(sys.executable)
print(os.listdir('.'))
if Path('./data').exists():
    print(os.listdir('./data'))

## Configure Paths

Update `PRED_PATH` and `GT_PATH` if your local folder layout is different. The path style below matches the repo-local layout used by the Depth Anything V2 notebook.

In [ ]:
REPO_DIR = Path.cwd()

# Preferred repo-local paths. Change these if your files live somewhere else.
PRED_PATH = REPO_DIR / 'data' / 'video_depth_anything' / 'video_depth_anything_depths.npy'
GT_PATH = REPO_DIR / 'data' / 'video_depth_anything' / 'groundtruth.npy'

# If you are reusing the same GT file from the Depth Anything V2 folder, uncomment this:
# GT_PATH = REPO_DIR / 'data' / 'depth_anything_v2' / 'groundtruth.npy'

# Example absolute paths, if needed:
# PRED_PATH = Path('/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/video_depth_anything/video_depth_anything_depths.npy')
# GT_PATH = Path('/Users/devrathod/Documents/classes_2026/Senior Design/Eco-Cars-Depth-Estimation-2026/data/depth_anything_v2/groundtruth.npy')

print('PRED_PATH:', PRED_PATH)
print('GT_PATH  :', GT_PATH)
print('PRED exists:', PRED_PATH.exists())
print('GT exists  :', GT_PATH.exists())

## Load Prediction and Ground Truth

In [ ]:
pred = np.load(PRED_PATH, mmap_mode='r')
gt = np.load(GT_PATH, mmap_mode='r')

print('pred shape:', pred.shape, pred.dtype)
print('gt shape  :', gt.shape, gt.dtype)

assert pred.ndim == 3, f'Expected pred shape [frames, height, width], got {pred.shape}'
assert gt.ndim == 3, f'Expected gt shape [frames, height, width], got {gt.shape}'
assert pred.shape[0] == gt.shape[0], f'Frame count mismatch: pred={pred.shape[0]}, gt={gt.shape[0]}'

## Calibration Helpers

In [ ]:
def resize_stack_to_gt(pred_stack, gt_shape):
    if pred_stack.shape[1:] == gt_shape:
        return np.asarray(pred_stack, dtype=np.float32)
    resized = [cv2.resize(np.asarray(frame, dtype=np.float32), (gt_shape[1], gt_shape[0]), interpolation=cv2.INTER_LINEAR) for frame in pred_stack]
    return np.stack(resized, axis=0).astype(np.float32)


def noise_handler(pred_values, gt_values):
    ground_truth = gt_values.flatten()
    predictions = pred_values.flatten()

    total_bins = 20
    bin_edges = np.linspace(predictions.min(), predictions.max(), total_bins + 1)
    bin_index = np.digitize(predictions, bin_edges) - 1
    bins_inlier = np.zeros(len(predictions), dtype=bool)

    for bin_id in range(total_bins):
        in_bin = bin_index == bin_id
        if in_bin.sum() < 25:
            continue
        gt_entry = ground_truth[in_bin]
        q1, q3 = np.percentile(gt_entry, [25, 75])
        iqr = q3 - q1
        bins_inlier[in_bin] = (gt_entry >= q1 - 1.5 * iqr) & (gt_entry <= q3 + 1.5 * iqr)

    p_clean = predictions[bins_inlier]
    g_clean = ground_truth[bins_inlier]
    poly = np.polyfit(p_clean, g_clean, deg=4)
    residuals = g_clean - np.polyval(poly, p_clean)
    zscores = np.abs(stats.zscore(residuals, nan_policy='omit'))
    keep = np.isfinite(zscores) & (zscores < 2.75)
    return p_clean[keep], g_clean[keep]


def model_hyperbolic(p, a, b, c):
    return a / (p + b) + c


def model_power(p, a, b, c):
    return a * np.power(np.clip(p, 1e-6, None), b) + c


def model_log(p, a, b, c):
    return a * np.log(np.clip(p + b, 1e-6, None)) + c


def model_exponential(p, a, b, c):
    return a * np.exp(b * p) + c


FN_MAP = {
    'hyperbolic': model_hyperbolic,
    'power': model_power,
    'log': model_log,
    'exponential': model_exponential,
}


def fit_all_models(p_clean, g_clean, n_sample=100_000):
    if len(p_clean) > n_sample:
        rng = np.random.default_rng(0)
        idx = rng.choice(len(p_clean), n_sample, replace=False)
        p_fit, g_fit = p_clean[idx], g_clean[idx]
    else:
        p_fit, g_fit = p_clean, g_clean

    models = {
        'hyperbolic': (model_hyperbolic, [-5.0, -0.85, 35.0], ([-np.inf, -1.0 + 1e-6, -np.inf], [np.inf, -1e-6, np.inf])),
        'power': (model_power, [10.0, 0.5, 0.0], ([-np.inf] * 3, [np.inf] * 3)),
        'log': (model_log, [-20.0, 1.0, 40.0], ([-np.inf] * 3, [np.inf] * 3)),
        'exponential': (model_exponential, [2.2, 2.97, 5.6], ([-np.inf] * 3, [np.inf] * 3)),
    }

    results = {}
    for name, (fn, p0, bounds) in models.items():
        try:
            popt, _ = curve_fit(fn, p_fit, g_fit, p0=p0, bounds=bounds, maxfev=50000)
            g_pred = fn(p_clean, *popt)
            rmse = np.sqrt(np.mean((g_clean - g_pred) ** 2))
            ss_res = np.sum((g_clean - g_pred) ** 2)
            ss_tot = np.sum((g_clean - g_clean.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot
            results[name] = {'params': popt, 'RMSE': rmse, 'R2': r2}
            print(f'{name:12s} R2={r2:.4f} RMSE={rmse:.4f}m params={np.round(popt, 4)}')
        except Exception as exc:
            print(f'{name}: failed - {exc}')
    return results


def compute_metrics(pred_eval, gt_eval, max_depth=40.0):
    mask = np.isfinite(pred_eval) & np.isfinite(gt_eval) & (gt_eval > 0.0) & (gt_eval <= max_depth)
    gt_v = gt_eval[mask]
    pred_v = np.clip(pred_eval[mask], 1e-6, None)

    absrel = np.mean(np.abs(pred_v - gt_v) / gt_v)
    rmse = np.sqrt(np.mean((pred_v - gt_v) ** 2))
    ratio = np.maximum(pred_v / gt_v, gt_v / pred_v)
    delta1 = np.mean(ratio < 1.25)
    return absrel, rmse, delta1, len(gt_v)

## Run Calibration and Evaluation

`START_INDEX = 10` matches the corrected Drive evaluation where frames `00000` through `00009` were excluded because their GT depth scale differs from the rest of the segment.

In [ ]:
START_INDEX = 10
CALIBRATION_FRAMES = 5
MAX_DEPTH = 40.0

pred_eval_stack = resize_stack_to_gt(pred[START_INDEX:], gt.shape[1:])
gt_eval_stack = np.asarray(gt[START_INDEX:], dtype=np.float32)

print('prediction stack:', pred_eval_stack.shape)
print('ground truth    :', gt_eval_stack.shape)

pred_calib = pred_eval_stack[:CALIBRATION_FRAMES]
gt_calib = gt_eval_stack[:CALIBRATION_FRAMES]

calib_mask = np.isfinite(pred_calib) & np.isfinite(gt_calib) & (gt_calib > 0.0) & (gt_calib <= MAX_DEPTH)
p_clean, g_clean = noise_handler(pred_calib[calib_mask], gt_calib[calib_mask])

print(f'Points after cleaning: {len(p_clean):,}')
fit_results = fit_all_models(p_clean, g_clean)

best_name = min(fit_results, key=lambda name: fit_results[name]['RMSE'])
best_params = fit_results[best_name]['params']
print('Selected model:', best_name)

pred_metric = FN_MAP[best_name](pred_eval_stack[CALIBRATION_FRAMES:], *best_params)
gt_metric = gt_eval_stack[CALIBRATION_FRAMES:]

absrel, rmse, delta1, n_pixels = compute_metrics(pred_metric, gt_metric, MAX_DEPTH)

print('\n=== FINAL VIDEO DEPTH ANYTHING METRICS ===')
print(f'Evaluation frames : frame_{START_INDEX + CALIBRATION_FRAMES:05d} through frame_{pred.shape[0] - 1:05d}')
print(f'Evaluation pixels : {n_pixels:,}')
print(f'AbsRel            : {absrel:.4f}')
print(f'RMSE              : {rmse:.4f} m')
print(f'delta1            : {delta1:.4f}')